# Notebook 01: EDA and Business Insights

**Project:** RevIQ AI: SaaS Revenue Intelligence Platform  
**Purpose:** Understand the data before modeling. Every chart here answers a business question first, a technical question second.

---

## What this notebook covers

1. Customer portfolio overview (who are our customers?)
2. Revenue distribution by segment and plan
3. Churn rate by segment and plan
4. Usage signals: how churned vs. healthy customers behave differently
5. Support signals: ticket volume, sentiment, resolution time
6. Customer success signals: NPS, health score, CS touchpoints
7. Renewal timing and its relationship with churn
8. Key business insights summary

**Prerequisites:** Run `python run_pipeline.py` from the project root before opening this notebook.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure we are running from the project root
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", "{:,.2f}".format)
print(f"Working directory: {Path.cwd()}")
print("Setup complete.")

In [ ]:
from src.config import SYNTHETIC_DIR, PROCESSED_DIR, REPORTS_DIR

customers    = pd.read_csv(SYNTHETIC_DIR / "customers.csv")
subscriptions = pd.read_csv(SYNTHETIC_DIR / "subscriptions.csv")
usage        = pd.read_csv(SYNTHETIC_DIR / "product_usage.csv")
support      = pd.read_csv(SYNTHETIC_DIR / "support.csv")
cs           = pd.read_csv(SYNTHETIC_DIR / "customer_success.csv")
targets      = pd.read_csv(SYNTHETIC_DIR / "targets.csv")

print(f"Customers:      {len(customers):>6,}")
print(f"Subscriptions:  {len(subscriptions):>6,}")
print(f"Usage records:  {len(usage):>6,}")
print(f"Support records:{len(support):>6,}")
print(f"CS records:     {len(cs):>6,}")
print(f"Target months:  {len(targets):>6,}")

In [ ]:
# Build one row per customer with their average signals and churn status.
# This is the foundation for all comparisons in this notebook.

ever_churned = (
    subscriptions.groupby("customer_id")["churned"]
    .max().reset_index()
    .rename(columns={"churned": "will_churn"})
)

fin_agg = subscriptions.groupby("customer_id").agg(
    avg_mrr=("mrr", "mean"),
    avg_arr=("arr", "mean"),
).reset_index()

usage_agg = usage.groupby("customer_id").agg(
    avg_logins=("logins", "mean"),
    avg_active_users=("active_users", "mean"),
    avg_feature_adoption=("feature_adoption", "mean"),
    avg_api_calls=("api_calls", "mean"),
).reset_index()

support_agg = support.groupby("customer_id").agg(
    avg_tickets=("tickets", "mean"),
    avg_sentiment=("sentiment", "mean"),
    avg_resolution_time=("avg_resolution_time", "mean"),
).reset_index()

cs_agg = cs.groupby("customer_id").agg(
    avg_nps=("nps", "mean"),
    avg_health=("health_score", "mean"),
    avg_last_touch=("last_touch_days", "mean"),
).reset_index()

snapshot = (
    customers
    .merge(ever_churned, on="customer_id")
    .merge(fin_agg,      on="customer_id")
    .merge(usage_agg,    on="customer_id")
    .merge(support_agg,  on="customer_id")
    .merge(cs_agg,       on="customer_id")
)
snapshot["churn_label"] = snapshot["will_churn"].map({0: "Healthy", 1: "Churned"})

SEGMENT_COLORS = {"SMB": "#2196F3", "Mid-Market": "#FF9800", "Enterprise": "#4CAF50"}
CHURN_COLORS   = {"Healthy": "#4CAF50", "Churned": "#f44336"}

print(f"Snapshot: {len(snapshot):,} customers")
print(f"Churned:  {snapshot['will_churn'].sum():,} ({snapshot['will_churn'].mean():.1%})")
print(f"Healthy:  {(snapshot['will_churn']==0).sum():,} ({(snapshot['will_churn']==0).mean():.1%})")

---
## 1. Customer Portfolio Overview

**Business question:** Who are our customers? What does the mix look like across segments, plans, countries, and industries?

This is the starting point for any business analysis. Before asking which customers churn, you need to know who your customers are.

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["By Segment", "By Plan", "By Country", "By Industry"]
)

seg_counts     = customers["segment"].value_counts().reindex(["SMB", "Mid-Market", "Enterprise"])
plan_counts    = customers["plan"].value_counts().reindex(["Starter", "Growth", "Professional", "Enterprise"])
country_counts = customers["country"].value_counts()
industry_counts = customers["industry"].value_counts()

for counts, row, col, colors in [
    (seg_counts,      1, 1, ["#2196F3", "#FF9800", "#4CAF50"]),
    (plan_counts,     1, 2, ["#9E9E9E", "#64B5F6", "#1976D2", "#0D47A1"]),
    (country_counts,  2, 1, None),
    (industry_counts, 2, 2, None),
]:
    bar_kwargs = dict(marker_color=colors) if colors else {}
    fig.add_trace(
        go.Bar(x=counts.index.tolist(), y=counts.values.tolist(), showlegend=False, **bar_kwargs),
        row=row, col=col
    )

fig.update_layout(title_text="Customer Portfolio Overview: 1,000 Customers", height=560)
fig.show()

**Key observations:**
- SMB is the largest segment (50%), Mid-Market is 35%, Enterprise is 15%
- Most customers are on Starter or Growth plans (the lower-revenue, higher-churn plans)
- The US is the dominant market (40%), followed by UK and Germany
- Customers span 6 industries, relatively balanced

This mix matters for churn analysis: **SMB customers on Starter plans are the highest-volume but also the highest-risk group.**

---
## 2. Revenue Distribution by Segment

**Business question:** How much ARR does each segment represent? Which customers are worth the most?

Not all customers are equal. Losing one Enterprise customer can wipe out the revenue of 20 SMB customers.

In [ ]:
fig = px.box(
    snapshot,
    x="segment", y="avg_arr",
    color="segment",
    color_discrete_map=SEGMENT_COLORS,
    category_orders={"segment": ["SMB", "Mid-Market", "Enterprise"]},
    title="ARR Distribution by Customer Segment",
    labels={"avg_arr": "Average ARR ($)", "segment": "Segment"},
    points="outliers",
)
fig.update_layout(yaxis_tickformat="$,.0f", showlegend=False, height=420)
fig.show()

arr_summary = snapshot.groupby("segment")["avg_arr"].agg(["median", "mean", "max"]).round(0)
arr_summary.columns = ["Median ARR", "Mean ARR", "Max ARR"]
arr_summary = arr_summary.reindex(["SMB", "Mid-Market", "Enterprise"])
print(arr_summary.applymap(lambda x: f"${x:,.0f}"))

**Key observation:** The ARR gap between segments is enormous. A median Enterprise customer is worth 20–40× a median SMB customer. This is why the Revenue Risk Score weights ARR size heavily: losing one Enterprise customer is a business event, not just a metric.

---
## 3. Churn Rate Analysis

**Business question:** Which segments and plans lose customers at the highest rate?

Here "churn rate" means: what fraction of customers in each group eventually churned during our 24-month observation window? This is a customer-level rate, not a monthly rate.

In [ ]:
churn_seg = (
    snapshot.groupby("segment")
    .agg(total=("customer_id", "count"), churned=("will_churn", "sum"))
    .reset_index()
)
churn_seg["churn_rate_pct"] = (churn_seg["churned"] / churn_seg["total"] * 100).round(1)
churn_seg["segment"] = pd.Categorical(
    churn_seg["segment"], categories=["SMB", "Mid-Market", "Enterprise"], ordered=True
)
churn_seg = churn_seg.sort_values("segment")

fig = px.bar(
    churn_seg, x="segment", y="churn_rate_pct",
    color="segment", color_discrete_map=SEGMENT_COLORS,
    text="churn_rate_pct",
    title="Customer Churn Rate by Segment (% of customers who churned)",
    labels={"churn_rate_pct": "Churn Rate (%)", "segment": "Segment"},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, height=420, yaxis_range=[0, 35])
fig.show()

In [ ]:
plan_order = ["Starter", "Growth", "Professional", "Enterprise"]
churn_plan = (
    snapshot.groupby("plan")
    .agg(total=("customer_id", "count"), churned=("will_churn", "sum"))
    .reset_index()
)
churn_plan["churn_rate_pct"] = (churn_plan["churned"] / churn_plan["total"] * 100).round(1)
churn_plan["plan"] = pd.Categorical(churn_plan["plan"], categories=plan_order, ordered=True)
churn_plan = churn_plan.sort_values("plan")

fig = px.bar(
    churn_plan, x="plan", y="churn_rate_pct",
    text="churn_rate_pct",
    title="Customer Churn Rate by Plan (% of customers who churned)",
    labels={"churn_rate_pct": "Churn Rate (%)", "plan": "Plan"},
    color_discrete_sequence=["#9C27B0"],
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, height=420, yaxis_range=[0, 35])
fig.show()

**Key observation:** Churn follows a clear inverse relationship with account size. SMB customers churn at a rate roughly 3–4× higher than Enterprise. Starter plan customers churn at the highest rate of any plan.

The implication: **a single retention strategy for all customers is wrong.** SMB needs automated, scalable low-touch retention. Enterprise needs high-touch, proactive CS.

---
## 4. Product Usage: Churned vs. Healthy Customers

**Business question:** Do customers who eventually churn use the product differently from those who stay?

If yes, those usage differences are what the ML model will learn to detect *before* the customer leaves.

In [ ]:
usage_metrics = {
    "avg_logins":           "Avg Monthly Logins",
    "avg_feature_adoption": "Avg Feature Adoption (0–1)",
    "avg_active_users":     "Avg Active Users",
    "avg_api_calls":        "Avg API Calls",
}

usage_compare = (
    snapshot.groupby("churn_label")[list(usage_metrics.keys())]
    .mean().reset_index()
)
usage_melted = usage_compare.melt(
    id_vars="churn_label",
    value_vars=list(usage_metrics.keys()),
    var_name="metric", value_name="avg_value"
)
usage_melted["metric_label"] = usage_melted["metric"].map(usage_metrics)

fig = px.bar(
    usage_melted, x="metric_label", y="avg_value", color="churn_label",
    barmode="group",
    title="Product Usage Signals: Healthy vs. Churned Customers (Customer Averages)",
    color_discrete_map=CHURN_COLORS,
    labels={"avg_value": "Average Value", "metric_label": "Metric", "churn_label": "Status"},
)
fig.update_layout(height=440, legend_title="Customer Status")
fig.show()

# Print the ratios
print("Healthy/Churned ratio by metric:")
for metric, label in usage_metrics.items():
    h = snapshot[snapshot["will_churn"]==0][metric].mean()
    c = snapshot[snapshot["will_churn"]==1][metric].mean()
    print(f"  {label}: healthy={h:.2f}, churned={c:.2f}, ratio={h/c:.1f}x")

**Key observation:** Healthy customers log in more, adopt more features, have more active users, and make more API calls than churned customers, across all metrics, by a significant margin.

Feature adoption is the strongest signal. A customer using a large fraction of available features has built workflows around your product. Leaving means rebuilding everything. A customer using 10% of features has almost no switching cost.

**This confirms the model's most important feature group: product usage trends.**

---
## 5. Support and Customer Success Signals

**Business question:** Do churning customers show different support and satisfaction patterns?

Support interactions and CS health scores are leading indicators: they change *before* the cancellation decision is made.

In [ ]:
signal_metrics = {
    "avg_tickets":         "Avg Monthly Tickets",
    "avg_sentiment":       "Avg Support Sentiment (0–1)",
    "avg_resolution_time": "Avg Resolution Time (days)",
    "avg_nps":             "Avg NPS (0–10)",
    "avg_health":          "Avg Health Score (0–1)",
    "avg_last_touch":      "Avg Days Since CS Touch",
}

signal_compare = (
    snapshot.groupby("churn_label")[list(signal_metrics.keys())]
    .mean().reset_index()
)
signal_melted = signal_compare.melt(
    id_vars="churn_label",
    value_vars=list(signal_metrics.keys()),
    var_name="metric", value_name="avg_value"
)
signal_melted["metric_label"] = signal_melted["metric"].map(signal_metrics)

fig = px.bar(
    signal_melted, x="metric_label", y="avg_value", color="churn_label",
    barmode="group",
    title="Support and CS Signals: Healthy vs. Churned Customers",
    color_discrete_map=CHURN_COLORS,
    labels={"avg_value": "Average Value", "metric_label": "Metric", "churn_label": "Status"},
)
fig.update_layout(height=440, legend_title="Customer Status",
                  xaxis_tickangle=-20)
fig.show()

print("Signal comparison: healthy vs. churned:")
for metric, label in signal_metrics.items():
    h = snapshot[snapshot["will_churn"]==0][metric].mean()
    c = snapshot[snapshot["will_churn"]==1][metric].mean()
    direction = "higher" if c > h else "lower"
    print(f"  {label}: healthy={h:.2f}, churned={c:.2f} (churned is {direction})")

**Key observations:**
- Churning customers open **more support tickets** but receive **lower sentiment** interactions: they are having more problems and feeling less satisfied with the responses
- Resolution time is longer for churning customers: their issues take more time to solve, adding to frustration
- NPS is significantly lower for churning customers (~4 vs. ~6)
- Health score is lower: the CS system is already flagging them
- **Days since last CS touch is the most actionable signal:** churning customers go much longer without hearing from anyone

The last point is critical for a CS team: it means **inaction is a predictor of churn.** Simply reaching out to at-risk customers can move the needle.

---
## 6. Renewal Timing and Churn Risk

**Business question:** Does churn happen more often when a contract renewal is approaching?

Contracts create a natural decision point. A dissatisfied customer might not cancel mid-contract, but they will not renew when the time comes.

In [ ]:
subs_copy = subscriptions.copy()
subs_copy["renewal_bucket"] = pd.cut(
    subs_copy["months_to_renewal"],
    bins=[0, 1, 3, 6, 12],
    labels=["≤ 1 month", "2–3 months", "4–6 months", "7–12 months"],
)

renewal_churn = (
    subs_copy.groupby("renewal_bucket", observed=True)
    .agg(total=("churned", "count"), churned=("churned", "sum"))
    .reset_index()
)
renewal_churn["churn_rate_pct"] = (
    renewal_churn["churned"] / renewal_churn["total"] * 100
).round(2)

fig = px.bar(
    renewal_churn, x="renewal_bucket", y="churn_rate_pct",
    text="churn_rate_pct",
    title="Monthly Churn Rate by Months to Contract Renewal",
    labels={"churn_rate_pct": "Monthly Churn Rate (%)", "renewal_bucket": "Months to Renewal"},
    color_discrete_sequence=["#E91E63"],
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(height=420, yaxis_range=[0, 6])
fig.show()

print("Churn rate by renewal window:")
print(renewal_churn[["renewal_bucket", "total", "churned", "churn_rate_pct"]].to_string(index=False))

**Key observation:** Monthly churn probability is significantly elevated when a contract renewal is within 1–3 months. This confirms that `months_to_renewal` is a valuable feature, not because it causes churn, but because it **defines when the risk materializes.**

For a CS team: a customer with bad signals AND a renewal in 6 weeks is a five-alarm fire. A customer with the same bad signals and a renewal in 10 months has time to be saved.

---
## 7. ARR Trend: Company Revenue Over Time

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=targets["month"], y=targets["actual_arr"] / 1e6,
    name="Actual ARR", mode="lines+markers",
    line=dict(color="#2196F3", width=2),
))
fig.add_trace(go.Scatter(
    x=targets["month"], y=targets["target_arr"] / 1e6,
    name="Target ARR", mode="lines",
    line=dict(color="#4CAF50", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=targets["month"], y=targets["forecast_arr"] / 1e6,
    name="Forecast ARR", mode="lines",
    line=dict(color="#FF9800", width=2, dash="dot"),
))
fig.update_layout(
    title="Company ARR Over 24 Months: Actual vs. Target vs. Forecast",
    xaxis_title="Month",
    yaxis_title="ARR ($M)",
    height=420,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

gap = targets["actual_arr"].iloc[-1] - targets["target_arr"].iloc[-1]
print(f"Latest month: Actual: ${targets['actual_arr'].iloc[-1]/1e6:.1f}M, "
      f"Target: ${targets['target_arr'].iloc[-1]/1e6:.1f}M, "
      f"Gap: ${gap/1e6:+.1f}M")

---
## 8. Key Business Insights Summary

Everything we found in this notebook that drives the modeling decisions:

---

### Finding 1: Churn is segment-driven
SMB customers churn at 3–4× the rate of Enterprise customers. Any churn prevention strategy must be segment-aware: what works for SMB (automated alerts, self-serve resources) will not work for Enterprise (executive calls, dedicated CS).

### Finding 2: Enterprise ARR concentration is extreme
The top 15% of customers (Enterprise) likely represent the majority of ARR. This means a small number of high-risk Enterprise customers should consume most of the CS team's retention effort, regardless of their raw churn probability.

### Finding 3: Product usage is the earliest churn signal
Churned customers have dramatically lower logins, feature adoption, and API calls throughout their lifetime, not just at the end. Usage decline is detectable months before cancellation, giving the CS team a meaningful intervention window.

### Finding 4: Support frustration is a compounding signal
Churning customers open more tickets but receive worse service (lower sentiment, longer resolution). This combination (more problems, less satisfaction) is a strong predictor. One bad support experience rarely causes churn; sustained frustration does.

### Finding 5: Inaction is a churn predictor
Days since last CS touchpoint is one of the clearest differentiators between healthy and churning customers. This is directly actionable: the CS team can change this number today by sending an email or scheduling a call.

### Finding 6: NPS below 5 is a warning flag
Churned customers have an average NPS around 4, healthy customers around 6. A drop from 7 to 4 over two quarters is worth investigating immediately, even if the customer hasn't opened a ticket.

### Finding 7: Renewal timing amplifies all other risk
Bad signals in a customer who renews in 8 months are manageable. The same signals in a customer who renews in 6 weeks are a five-alarm fire. The model's Revenue Risk Score accounts for this by weighting renewal urgency at 20%.

### Finding 8: The model's job is pattern detection at scale
A human CS manager can look at 20 accounts and make good judgment calls. The model looks at 1,000 accounts and does the same, consistently, every month. Its value is not replacing human judgment: it is making human judgment scalable.

---

**Next:** See `notebooks/02_model_evaluation.ipynb` for how these patterns translate into a trained churn prediction model.